In [1]:
#parsing PDF

In [10]:
from langchain_community.document_loaders import ( 
    PyPDFLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader
 )

C:\Users\Amrose\AppData\Local\Temp\ipykernel_28972\2293025773.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [11]:
print("pypdf loader")

try:
    Pypdf_loader = PyPDFLoader("doc/sam.pdf")
    pypdf_docs=Pypdf_loader.load()
    print(pypdf_docs)
except Exception as e:
    print(f"error: {e}")

pypdf loader
[Document(metadata={'producer': 'OpenPDF 1.0.0-SNAPSHOT; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'PyPDF', 'creationdate': '2025-06-07T06:24:41+00:00', 'meeting starting date': '28 March 2025', 'moddate': '2025-08-07T23:26:44-04:00', 'ieee article id': '11106054', 'ieee issue id': '11105581', 'subject': '2025 11th International Conference on Computing and Artificial Intelligence (ICCAI);2025; ; ;10.1109/ICCAI66501.2025.00018', 'ieee publication id': '11105578', 'title': 'Advancing Industrial Safety: A Spatio-Temporal Framework for PPE Detection Using YOLOv11', 'meeting ending date': '31 March 2025', 'source': 'doc/sam.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='Advancing Industrial Safety: A Spatio-Temporal \nFramework for PPE Detection Using YOLOv11 \nTeeraphat Inta1,*, Choosak Pornsing1 \n1 Department of Industrial Engineering and Management, Faculty of Engineering and Industrial Technology, \nSilpako

In [12]:
#method2:pymupdf loader
print("pymupdf loader")
try:

    pymupdf_loader = PyMuPDFLoader("doc/sam.pdf")
    pymupdf_docs = pymupdf_loader.load()
    print(f"Loaded {len(pymupdf_docs)} pages")
    print(pymupdf_docs)
except Exception as e:
    print(f"error: {e}")    

pymupdf loader
Loaded 5 pages
[Document(metadata={'producer': 'OpenPDF 1.0.0-SNAPSHOT; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': '', 'creationdate': '2025-06-07T06:24:41+00:00', 'source': 'doc/sam.pdf', 'file_path': 'doc/sam.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'Advancing Industrial Safety: A Spatio-Temporal Framework for PPE Detection Using YOLOv11', 'author': '', 'subject': '2025 11th International Conference on Computing and Artificial Intelligence (ICCAI);2025; ; ;10.1109/ICCAI66501.2025.00018', 'keywords': '', 'moddate': '2025-08-07T23:26:44-04:00', 'trapped': '', 'modDate': "D:20250807232644-04'00'", 'creationDate': 'D:20250607062441Z', 'page': 0}, page_content='Advancing Industrial Safety: A Spatio-Temporal \nFramework for PPE Detection Using YOLOv11 \nTeeraphat Inta1,*, Choosak Pornsing1 \n1 Department of Industrial Engineering and Management, Faculty of Engineering and Industrial Technology, \nSilpakorn \nUiversity,

Handlng issues in pdf


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [27]:
from langchain_core.documents import Document
from typing import List
class PDFprocessor:
    def __init__(self,chunk_size=1000, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=[" "]

            )
    def process_pdf(self,pdf_path:str)->List[Document]:
        loader =PyPDFLoader(pdf_path)
        pages = loader.load()

        processed_chunks = []

        for page_num,page in enumerate(pages):
            cleaned_text = self.clean_text(page.page_content)

            if len(cleaned_text.strip()) < 50:
                continue    


            chunks =self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "PDFprocessor",
                    "char_count": len(cleaned_text)
                    
                    
                    
                    
                    }]
            )
            processed_chunks.extend(chunks)
        return processed_chunks
    
    def clean_text(self,text:str)->str:
        #remove unwanted characters and whitespace
        text ="".join(text.split())


        text =text.replace("fi","fi")
        text =text.replace("fl","fl")

        return text

In [28]:
preprocessor= PDFprocessor()

In [29]:
preprocessor


In [30]:
### try if pdf is available and process it

try:
    pro_chunks = preprocessor.process_pdf("doc/sam.pdf")
    print(f"Processed {len(pro_chunks)} chunks")

    if pro_chunks:
        print("\nSample chunk:")
        for key,value in pro_chunks[0].metadata.items():
            print(f"{key}: {value}")

except Exception as e:
    print(f"error: {e}")




Processed 5 chunks

Sample chunk:
producer: OpenPDF 1.0.0-SNAPSHOT; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV
creator: PyPDF
creationdate: 2025-06-07T06:24:41+00:00
meeting starting date: 28 March 2025
moddate: 2025-08-07T23:26:44-04:00
ieee article id: 11106054
ieee issue id: 11105581
subject: 2025 11th International Conference on Computing and Artificial Intelligence (ICCAI);2025; ; ;10.1109/ICCAI66501.2025.00018
ieee publication id: 11105578
title: Advancing Industrial Safety: A Spatio-Temporal Framework for PPE Detection Using YOLOv11
meeting ending date: 31 March 2025
source: doc/sam.pdf
total_pages: 5
page: 1
page_label: 1
chunk_method: PDFprocessor
char_count: 4243
